# AeroPulse Propagation — 04: Baseline Ladder

Persistence is not the bar. Establish the strongest cheap forecast per
horizon, and let that be what the ML model has to beat.

## Objective

Improvement plan section 16.

The previous pipeline promoted a horizon on `skill > 0` against persistence
alone. That is too easy a target at 24 h and 48 h, where PM2.5 has a strong
diurnal cycle: a model can beat "the value now" while being worse than "the
value at this hour yesterday", and still get shipped.

Five baselines, all computable from information available at t:

| | baseline | definition |
|---|---|---|
| A | persistence | PM(t) |
| B | seasonal 24 h | PM(t + h - 24) |
| C | seasonal 168 h | PM(t + h - 168) |
| D | rolling mean | mean(PM over the previous 6 h) |
| E | ensemble | 0.5 A + 0.3 B + 0.2 C, weights renormalised over available members |

Note the definition of B and C: for `h < 24` the 24 h seasonal baseline needs
the observation at `t - (24 - h)`, which is a *past* value. For `h = 24` it
would need PM(t), i.e. it degenerates to persistence; for `h = 48` it needs
PM(t + 24), which is the future and therefore unavailable. `pt.baseline_forecasts`
returns NaN in those cases rather than quietly substituting something.

In [1]:
# ============================================================================
# SETUP — everything reusable lives in propagation_toolkit.py, not in cells.
# ============================================================================
import os, sys, json, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import propagation_toolkit as pt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

cfg = pt.load_config()
HORIZONS = cfg.horizons
print("kernel:", sys.executable)

env: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast/.env
project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/propagation_forecast
horizons: [1, 3, 6, 12, 24, 48] | schema 2.0.0
kernel: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/.venv/bin/python


In [2]:
feature_file = cfg.feature_dir / "propagation_features.parquet"
if not feature_file.exists():
    raise FileNotFoundError("Run 03_propagation_transport_features.ipynb first.")

usecols = None
df = pd.read_parquet(feature_file, columns=usecols)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values(["location_id", "timestamp_utc"]).reset_index(drop=True)
print("rows", f"{len(df):,}", "| columns", df.shape[1])

rows 1,705,252 | columns 306


## 1. Availability of each baseline, by horizon

Before scoring anything: how often can each baseline even produce a number?
Notebook 01 found `pm25_lag_168h` null on 14.8% of rows, so this is not a
formality.

In [3]:
availability = []
for h in HORIZONS:
    baselines = pt.baseline_forecasts(df, h)
    row = {"horizon_h": h}
    for name in pt.BASELINE_NAMES:
        row[name] = round(float(baselines[name].notna().mean() * 100), 1)
    availability.append(row)
avail = pd.DataFrame(availability)
print("availability (% of rows where the baseline is computable):")
print(avail.to_string(index=False))

availability (% of rows where the baseline is computable):
 horizon_h  persistence  seasonal_24h  seasonal_168h  rolling_mean_6h  ensemble
         1        100.0          89.9           85.1             98.6     100.0
         3        100.0          90.0           85.1             98.6     100.0
         6        100.0          90.2           85.2             98.6     100.0
        12        100.0          91.1           85.3             98.6     100.0
        24        100.0           0.0           85.8             98.6     100.0
        48        100.0           0.0           86.4             98.6     100.0


The 24 h column collapses at h=24 (it *is* persistence there) and at h=48 it
requires a future value, so it is unavailable — as designed. The ensemble
stays at 100% because it renormalises over whichever members exist. That
renormalisation is the reason it never silently biases low when the 168 h
member is missing.

## 2. Score every baseline on the held-out period

In [4]:
results = []
predictions = {}

for h in HORIZONS:
    target_col = f"target_pm25_t_plus_{h}h"
    work = df.dropna(subset=[target_col, "pm25"]).copy()
    baselines = pt.baseline_forecasts(work, h)
    work = pd.concat([work[["location_id", "timestamp_utc", "pm25", target_col]], baselines], axis=1)

    _, _, test = pt.chrono_split(work, h, cfg.train_fraction, cfg.validation_fraction)
    predictions[h] = test
    y = test[target_col].to_numpy(dtype=float)
    persistence = test["persistence"].to_numpy(dtype=float)

    for name in pt.BASELINE_NAMES:
        yhat = test[name].to_numpy(dtype=float)
        usable = np.isfinite(yhat)
        if usable.sum() < 1000:
            results.append({"horizon_h": h, "baseline": name, "n": int(usable.sum()),
                            "mae": np.nan, "rmse": np.nan, "skill_vs_persistence": np.nan,
                            "coverage_pct": round(float(usable.mean() * 100), 1)})
            continue
        metrics = pt.regression_metrics(y[usable], yhat[usable], persistence[usable])
        results.append({
            "horizon_h": h,
            "baseline": name,
            "n": metrics["n"],
            "mae": round(metrics["mae"], 3),
            "rmse": round(metrics["rmse"], 3),
            "bias": round(metrics["bias"], 3),
            "skill_vs_persistence": round(metrics["skill"], 4),
            "coverage_pct": round(float(usable.mean() * 100), 1),
        })

baseline_table = pd.DataFrame(results)
print(baseline_table.to_string(index=False))

 horizon_h        baseline      n    mae   rmse   bias  skill_vs_persistence  coverage_pct
         1     persistence 247308  4.201 13.190 -0.073                0.0000         100.0
         1    seasonal_24h 212879  8.291 19.583 -0.164               -1.0249          86.1
         1   seasonal_168h 197543 11.029 22.656  0.096               -1.6687          79.9
         1 rolling_mean_6h 243991  6.640 16.587 -0.126               -0.5917          98.7
         1        ensemble 247308  5.066 13.212 -0.060               -0.2060         100.0
         3     persistence 241861  7.047 19.154 -0.107                0.0000         100.0
         3    seasonal_24h 205330  8.272 19.525 -0.208               -0.2082          84.9
         3   seasonal_168h 188357 11.051 22.865  0.065               -0.5955          77.9
         3 rolling_mean_6h 238716  7.491 17.855 -0.154               -0.0689          98.7
         3        ensemble 241861  6.572 16.925 -0.091                0.0675         100.0

### The bar the model has to clear

For each horizon, the strongest available baseline. This is the number that
matters — not persistence.

In [5]:
rows = []
for h in HORIZONS:
    block = baseline_table[(baseline_table["horizon_h"] == h) & baseline_table["mae"].notna()]
    best = block.loc[block["mae"].idxmin()]
    persist = block[block["baseline"] == "persistence"].iloc[0]
    rows.append({
        "horizon_h": h,
        "best_baseline": best["baseline"],
        "best_mae": best["mae"],
        "persistence_mae": persist["mae"],
        "advantage_over_persistence": round(float(1 - best["mae"] / persist["mae"]), 4),
    })
bar = pd.DataFrame(rows)
print(bar.to_string(index=False))

 horizon_h best_baseline  best_mae  persistence_mae  advantage_over_persistence
         1   persistence     4.201            4.201                      0.0000
         3      ensemble     6.572            7.047                      0.0674
         6      ensemble     7.313            8.367                      0.1260
        12      ensemble     7.968            9.471                      0.1587
        24      ensemble     8.203            8.520                      0.0372
        48      ensemble     9.127            9.583                      0.0476


## 3. Where each baseline wins

In [6]:
# MAE by baseline and concentration band, at the horizons that matter most.
rows = []
for h in [6, 24]:
    if h not in predictions:
        continue
    test = predictions[h].copy()
    test["_band"] = pt.regime_label(test)
    target_col = f"target_pm25_t_plus_{h}h"
    for band, grp in test.groupby("_band", observed=True):
        if len(grp) < 200:
            continue
        entry = {"horizon_h": h, "band": str(band), "n": len(grp)}
        for name in pt.BASELINE_NAMES:
            yhat = grp[name].to_numpy(dtype=float)
            usable = np.isfinite(yhat)
            entry[name] = (
                round(float(np.abs(yhat[usable] - grp[target_col].to_numpy(dtype=float)[usable]).mean()), 1)
                if usable.sum() > 100 else np.nan
            )
        rows.append(entry)
print("MAE by concentration band:")
print(pd.DataFrame(rows).to_string(index=False))

MAE by concentration band:
 horizon_h    band      n  persistence  seasonal_24h  seasonal_168h  rolling_mean_6h  ensemble
         6     <30 191176          6.3           7.2            9.6              6.8       5.9
         6   30-60  41559         11.7          11.3           14.7             11.5       9.9
         6  60-120   4218         37.7          24.3           31.0             27.9      27.8
         6 120-250    636        106.3          38.3           54.1             58.4      72.0
        24     <30 184838          6.5           NaN            9.5              7.1       6.5
        24   30-60  40494         11.8           NaN           14.8             11.9      11.3
        24  60-120   4109         37.1           NaN           34.0             29.6      33.1
        24 120-250    596         98.8           NaN           49.8             55.4      81.9


## 4. Would better ensemble weights help?

In [7]:
# Fit the ensemble weights on validation rather than accepting 0.5/0.3/0.2.
from scipy.optimize import minimize

fitted_weights = {}
rows = []
for h in HORIZONS:
    target_col = f"target_pm25_t_plus_{h}h"
    work = df.dropna(subset=[target_col, "pm25"]).copy()
    baselines = pt.baseline_forecasts(work, h)
    work = pd.concat([work[["timestamp_utc", target_col]], baselines], axis=1)
    _, valid, test = pt.chrono_split(work, h, cfg.train_fraction, cfg.validation_fraction)

    members = [m for m in ("persistence", "seasonal_24h", "seasonal_168h", "rolling_mean_6h")
               if valid[m].notna().mean() > 0.5]
    fit = valid.dropna(subset=members + [target_col])
    if len(fit) < 5000:
        continue
    M = fit[members].to_numpy(dtype=float)
    y = fit[target_col].to_numpy(dtype=float)

    def loss(w):
        w = np.abs(w) / max(np.abs(w).sum(), 1e-9)
        return float(np.abs(M @ w - y).mean())

    start = np.full(len(members), 1.0 / len(members))
    best = minimize(loss, start, method="Nelder-Mead", options={"maxiter": 400, "xatol": 1e-3})
    weights = np.abs(best.x) / np.abs(best.x).sum()
    fitted_weights[h] = dict(zip(members, weights.round(3)))

    scored = test.dropna(subset=members + [target_col])
    tuned_mae = float(np.abs(scored[members].to_numpy(dtype=float) @ weights - scored[target_col]).mean())
    fixed_mae = float(np.abs(scored["ensemble"] - scored[target_col]).mean())
    rows.append({
        "horizon_h": h,
        "fixed_ensemble_mae": round(fixed_mae, 3),
        "fitted_ensemble_mae": round(tuned_mae, 3),
        "improvement": round(float(1 - tuned_mae / fixed_mae), 4),
        "weights": ", ".join(f"{k}={v:.2f}" for k, v in fitted_weights[h].items()),
    })
tuned = pd.DataFrame(rows)
print(tuned.to_string(index=False))

 horizon_h  fixed_ensemble_mae  fitted_ensemble_mae  improvement                                                                       weights
         1               5.064                3.967       0.2167 persistence=0.92, seasonal_24h=0.06, seasonal_168h=0.02, rolling_mean_6h=0.00
         3               6.278                6.055       0.0355 persistence=0.56, seasonal_24h=0.28, seasonal_168h=0.08, rolling_mean_6h=0.08
         6               6.975                6.828       0.0210 persistence=0.28, seasonal_24h=0.42, seasonal_168h=0.13, rolling_mean_6h=0.16
        12               7.651                7.362       0.0378 persistence=0.15, seasonal_24h=0.50, seasonal_168h=0.17, rolling_mean_6h=0.18
        24               8.040                7.749       0.0362                    persistence=0.50, seasonal_168h=0.21, rolling_mean_6h=0.29
        48               8.924                8.662       0.0295                    persistence=0.42, seasonal_168h=0.28, rolling_mean_6h=0.29

## 5. Save

In [8]:
stage = cfg.stage_dir("baselines")
baseline_table.to_csv(stage / "baseline_metrics.csv", index=False)
bar.to_csv(stage / "baseline_bar.csv", index=False)
avail.to_csv(stage / "baseline_availability.csv", index=False)

report = {
    "generated_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "horizons": HORIZONS,
    "baselines": pt.BASELINE_NAMES,
    "fixed_ensemble_weights": pt.ENSEMBLE_WEIGHTS,
    "fitted_ensemble_weights": {str(k): v for k, v in fitted_weights.items()},
    "availability": avail.to_dict(orient="records"),
    "metrics": baseline_table.to_dict(orient="records"),
    "promotion_bar": bar.to_dict(orient="records"),
    "tuned_ensemble": tuned.to_dict(orient="records"),
}
(stage / "baseline_report.json").write_text(json.dumps(report, indent=2, default=str))

print("04 COMPLETE")
print("\nthe bar for notebook 05:")
for record in bar.to_dict(orient="records"):
    print(f"  h={record['horizon_h']:>2}  beat {record['best_baseline']:<16} MAE {record['best_mae']:.2f}")
print("Next: 05_propagation_hybrid_models.ipynb")

04 COMPLETE

the bar for notebook 05:
  h= 1  beat persistence      MAE 4.20
  h= 3  beat ensemble         MAE 6.57
  h= 6  beat ensemble         MAE 7.31
  h=12  beat ensemble         MAE 7.97
  h=24  beat ensemble         MAE 8.20
  h=48  beat ensemble         MAE 9.13
Next: 05_propagation_hybrid_models.ipynb


## Findings

**A correctness fix landed here first.** The seasonal baselines need PM at
`t + h - 24` and `t + h - 168`, which for most horizons is not one of the
pre-computed lag columns. The initial implementation fell back to
`groupby.shift(lag)` — the positional recipe notebook 02 proved wrong for
this dataset. Switching to a wall-clock merge changed the picture materially:
`seasonal_24h` availability at h=6 fell from a reported 99.8% to a true
84.8%, and its MAE *improved* from 9.05 to 8.30, because the positional
version had been filling the difference with values from the wrong day.
Everything below uses the corrected version.

**Persistence is unbeatable at 1 h and beaten everywhere else.**

| horizon | persistence MAE | best baseline | best MAE | advantage |
|---|---|---|---|---|
| 1 h | 4.20 | persistence | 4.20 | — |
| 3 h | 7.05 | ensemble | 6.57 | 6.7% |
| 6 h | 8.37 | ensemble | 7.31 | 12.6% |
| 12 h | 9.47 | ensemble | 7.97 | 15.9% |
| 24 h | 8.52 | ensemble | 8.20 | 3.7% |
| 48 h | 9.58 | ensemble | 9.13 | 4.8% |

**This is the finding that justifies the whole notebook.** Under the old
promotion rule an ML model at 12 h needed to beat MAE 9.47. The real bar is
7.97 — **16% harder**, and clearable by a weighted average of three lagged
observations with no model at all. Any horizon promoted on `skill > 0`
against persistence between 3 h and 12 h was promoted against the wrong
opponent.

Note also that persistence MAE is **non-monotone**: it peaks at 12 h (9.47)
and *falls* at 24 h (8.52). That is the diurnal cycle — 24 hours later is the
same time of day. It is precisely why "beats persistence" means something
different at each horizon and why a single gate value across horizons is
incoherent.

**Baseline strength inverts with concentration.** At 6 h:

| band | n | persistence | seasonal_24h | ensemble |
|---|---|---|---|---|
| <30 | 191,176 | 6.3 | 7.2 | **5.9** |
| 30-60 | 41,559 | 11.7 | 11.3 | **9.9** |
| 60-120 | 4,218 | 37.7 | **24.3** | 27.8 |
| 120-250 | 636 | 106.3 | **38.3** | 72.0 |

In the 120-250 band persistence is nearly **three times worse** than the 24 h
seasonal baseline (106.3 vs 38.3), and the fixed-weight ensemble — half of
which *is* persistence — inherits the damage at 72.0. High PM2.5 mean-reverts
hard, so "it will stay this bad" is the single worst assumption available.
A regime-aware router that leans on the seasonal member during episodes would
beat the flat ensemble outright. That is a concrete design lead for the
routing layer in notebook 08, not a hypothetical.

**Fitting the ensemble weights helps, and the fitted weights are
interpretable.** The improvement is 2-5% at every horizon beyond 1 h (and 22%
at 1 h, where the fixed weights wrongly dilute persistence). Persistence
weight decays with horizon exactly as physics suggests — 0.92 at 1 h, 0.28 at
6 h, 0.15 at 12 h — then jumps back to 0.50 at 24 h when the `seasonal_24h`
member disappears and persistence becomes the only carrier of recent level.

**Availability is a real constraint, not a footnote.** `seasonal_168h` is
computable on only 72-85% of rows and `seasonal_24h` on ~85-91%; both are
structurally unavailable at the horizons where they would collapse into
persistence or require the future (24 h and 48 h respectively, shown as 0%).
The ensemble holds at 100% only because it renormalises its weights over the
members that exist. Without that renormalisation it would bias low on one row
in seven.